# 02 - SARIMA: Modelo ARIMA com Componente Sazonal

Este notebook apresenta o modelo **SARIMA(p,d,q)(P,D,Q)[s]**, a extensao sazonal do ARIMA.

**Conteudo:**
1. Teoria do SARIMA
2. Decomposicao sazonal do airline.csv
3. Ajuste SARIMA com chronobox
4. Comparacao com/sem componente sazonal
5. Exercicios

## 1. Fundamentacao Teorica

O modelo **SARIMA(p,d,q)(P,D,Q)[s]** estende o ARIMA adicionando termos sazonais:

$$\Phi_P(L^s) \, \phi_p(L) \, (1 - L)^d \, (1 - L^s)^D \, y_t = c + \Theta_Q(L^s) \, \theta_q(L) \, \varepsilon_t$$

Onde:
- $(p, d, q)$: parte nao-sazonal (AR, diferenciacao, MA)
- $(P, D, Q)$: parte sazonal (AR sazonal, diferenciacao sazonal, MA sazonal)
- $s$: periodo sazonal (12 para dados mensais, 4 para trimestrais)
- $L^s$: operador de defasagem sazonal, $L^s y_t = y_{t-s}$

**Exemplo classico:** SARIMA(0,1,1)(0,1,1)[12] para dados mensais de passageiros aereos (Box-Jenkins airline model).

## 2. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox import ARIMA, ClassicalDecomposition
from chronobox.tests_stat import adf_test, ljung_box_test
from chronobox.visualization import (
    plot_diagnostics, plot_forecast, plot_series, plot_decomposition, set_theme
)

set_theme('professional')
np.random.seed(42)

print('chronobox importado com sucesso!')

In [ ]:
# Carregar dataset airline (passageiros aereos mensais, 1949-1960)
import os
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

airline = pd.read_csv(os.path.join(DATA_DIR, 'airline.csv'), parse_dates=['date'])
airline.set_index('date', inplace=True)

y = airline['passengers'].values

print(f'Periodo: {airline.index[0]} a {airline.index[-1]}')
print(f'Observacoes: {len(y)}')
print(f'Frequencia: mensal (s=12)')
airline.head()

In [ ]:
# Grafico 1: Serie temporal com tendencia e sazonalidade visiveis
fig = plot_series(y, labels=['Passageiros'], title='Passageiros Aereos Internacionais (1949-1960)')
plt.xlabel('Mes')
plt.ylabel('Milhares de passageiros')
plt.show()

print('Observa-se: tendencia crescente + sazonalidade multiplicativa (amplitude cresce com nivel)')

## 3. Decomposicao Sazonal

Antes de ajustar o SARIMA, vamos decompor a serie para entender seus componentes.

In [ ]:
# Decomposicao classica multiplicativa
decomp = ClassicalDecomposition(period=12, model='multiplicative')
result_decomp = decomp.fit(y)

# Grafico 2: Decomposicao sazonal (4 paineis)
fig = plot_decomposition(result_decomp, title='Decomposicao Multiplicativa - Airline')
plt.show()

print('Componentes identificados:')
print('- Tendencia: crescimento quase linear')
print('- Sazonalidade: pico no verao (jul-ago), vale no inverno (nov-fev)')
print('- Residuo: variancia relativamente constante apos remocao de tendencia e sazonalidade')

In [ ]:
# Transformacao log para estabilizar variancia (sazonalidade multiplicativa)
y_log = np.log(y)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y, color='steelblue')
axes[0].set_title('Serie Original')
axes[1].plot(y_log, color='darkorange')
axes[1].set_title('Serie em Log')
plt.tight_layout()
plt.show()

print('A transformacao log estabiliza a variancia: amplitude sazonal fica constante.')

## 4. Identificacao da Ordem do SARIMA

Para dados mensais com sazonalidade:
1. Aplicar diferenciacao sazonal ($D=1$): $\Delta_{12} y_t = y_t - y_{t-12}$
2. Se necessario, diferenciacao regular ($d=1$): $\Delta \Delta_{12} y_t$
3. Analisar ACF/PACF dos residuos

In [ ]:
# Diferenciacao sazonal e regular
y_sdiff = y_log[12:] - y_log[:-12]  # D=1 (diferenca sazonal)
y_ddiff = np.diff(y_sdiff)           # d=1 (diferenca regular)

# Testar estacionariedade
adf_sdiff = adf_test(y_sdiff)
adf_ddiff = adf_test(y_ddiff)

print(f'Apos D=1: ADF p-valor = {adf_sdiff.pvalue:.4f} -> {"Estacionaria" if adf_sdiff.pvalue < 0.05 else "Nao estacionaria"}')
print(f'Apos D=1, d=1: ADF p-valor = {adf_ddiff.pvalue:.4f} -> {"Estacionaria" if adf_ddiff.pvalue < 0.05 else "Nao estacionaria"}')

## 5. Ajuste SARIMA com chronobox

In [ ]:
# Modelo classico de Box-Jenkins: SARIMA(0,1,1)(0,1,1)[12]
sarima = ARIMA(order=(0, 1, 1), seasonal_order=(0, 1, 1, 12))
res_sarima = sarima.fit(y_log)

print(res_sarima.summary())

In [ ]:
# Grafico 3: Diagnostico dos residuos do SARIMA
fig = plot_diagnostics(res_sarima, lags=24, title='Diagnostico - SARIMA(0,1,1)(0,1,1)[12]')
plt.show()

In [ ]:
# Ljung-Box nos residuos do SARIMA
resid = res_sarima.residuals
for lag in [12, 24, 36]:
    lb = ljung_box_test(resid, lags=lag, model_df=2)
    print(f'Ljung-Box(lag={lag:2d}): Q={lb.statistic:8.3f}, p={lb.pvalue:.4f} -> {"OK" if lb.pvalue > 0.05 else "Autocorrelacao residual"}')

## 6. Comparacao: com vs sem Componente Sazonal

Vamos comparar o SARIMA com um ARIMA puro para demonstrar a importancia do componente sazonal.

In [ ]:
# Modelo SEM sazonalidade
arima_puro = ARIMA(order=(1, 1, 1))
res_arima = arima_puro.fit(y_log)

# Modelo COM sazonalidade
sarima_011_011 = ARIMA(order=(0, 1, 1), seasonal_order=(0, 1, 1, 12))
res_sarima_011 = sarima_011_011.fit(y_log)

# Variantes SARIMA
sarima_110_011 = ARIMA(order=(1, 1, 0), seasonal_order=(0, 1, 1, 12))
res_sarima_110 = sarima_110_011.fit(y_log)

sarima_111_011 = ARIMA(order=(1, 1, 1), seasonal_order=(0, 1, 1, 12))
res_sarima_111 = sarima_111_011.fit(y_log)

print(f'{"Modelo":<35} {"AIC":>10} {"BIC":>10}')
print('-' * 57)
for nome, res in [
    ('ARIMA(1,1,1) - sem sazonal', res_arima),
    ('SARIMA(0,1,1)(0,1,1)[12]', res_sarima_011),
    ('SARIMA(1,1,0)(0,1,1)[12]', res_sarima_110),
    ('SARIMA(1,1,1)(0,1,1)[12]', res_sarima_111),
]:
    print(f'{nome:<35} {res.aic:>10.2f} {res.bic:>10.2f}')

print('\nConclusion: O SARIMA supera amplamente o ARIMA puro.')
print('A componente sazonal captura padroes que o ARIMA ignora.')

In [ ]:
# Grafico 4: Previsao ARIMA vs SARIMA (12 meses)
fc_arima = res_arima.forecast(steps=12)
fc_sarima = res_sarima_011.forecast(steps=12)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ARIMA puro
n = len(y_log)
axes[0].plot(range(n), y_log, color='steelblue', label='Observado')
axes[0].plot(range(n, n+12), fc_arima['forecast'], color='red', linewidth=2, label='Previsao')
axes[0].fill_between(range(n, n+12), fc_arima['lower'], fc_arima['upper'], alpha=0.2, color='red')
axes[0].set_title('ARIMA(1,1,1) - Sem Sazonalidade')
axes[0].legend()

# SARIMA
axes[1].plot(range(n), y_log, color='steelblue', label='Observado')
axes[1].plot(range(n, n+12), fc_sarima['forecast'], color='darkorange', linewidth=2, label='Previsao')
axes[1].fill_between(range(n, n+12), fc_sarima['lower'], fc_sarima['upper'], alpha=0.2, color='darkorange')
axes[1].set_title('SARIMA(0,1,1)(0,1,1)[12]')
axes[1].legend()

plt.suptitle('Comparacao: Previsao com e sem Sazonalidade (log-escala)', y=1.02)
plt.tight_layout()
plt.show()

print('O SARIMA produz previsoes com padrao sazonal; o ARIMA puro gera previsoes monotona.')

In [ ]:
# Grafico 5: Previsao final do melhor modelo (escala original)
fig = plot_forecast(res_sarima_011, steps=24, alpha=0.05,
                    title='Previsao SARIMA(0,1,1)(0,1,1)[12] - 24 meses')
plt.show()

## Exercicio

### Exercicio 1: Ajustar SARIMA no brazil_ipca.csv

1. Carregue o dataset `brazil_ipca.csv` (inflacao mensal brasileira)
2. Faca a decomposicao sazonal (aditiva ou multiplicativa?)
3. Identifique a ordem via ACF/PACF
4. Ajuste pelo menos 3 modelos SARIMA com s=12
5. Compare por AIC e selecione o melhor
6. Faca previsao de 12 meses e plote o resultado

In [ ]:
# Exercicio 1 - Seu codigo aqui
# ipca = pd.read_csv(os.path.join(DATA_DIR, 'brazil_ipca.csv'), parse_dates=['date'])
# ...

## Exercicio

### Exercicio 2: Efeito da transformacao log

1. Ajuste SARIMA(0,1,1)(0,1,1)[12] no airline.csv SEM transformacao log
2. Compare os residuos (homocedasticidade) com o modelo em log
3. Qual abordagem produz melhores diagnosticos?

In [ ]:
# Exercicio 2 - Seu codigo aqui
# ...